# Tailored Learning for Cost Modeling: An Open Source RAG-Based Tool
Ryan Bell ryan.bell@nps.edu <br>
Ryan Longshore ryan.longshore@nps.edu <br>
Raymond Madachy, PhD rjmadach@nps.edu


Abstract:

For professionals entering the specialized field of cost modeling, there is often a significant gap in domain-specific knowledge. To address this, we present an AI-driven approach that leverages Python’s Haystack package to employ a Retrieval-Augmented Generation (RAG) pipeline to offer customized interactions with foundational cost modeling resources, including Barry Boehm’s books and other prominent cost modeling documents. This presentation explores the integration of this AI tool to deliver on-demand, conversational access to cost modeling knowledge, as well as a secondary feature for generating multiple-choice questions to reinforce learning.



The RAG pipeline enables engineers to query specific documents and receive relevant information from established cost modeling tools, simulating a mentor-mentee interaction and providing a much-needed knowledge repository. By dynamically generating both responses to complex inquiries and formative assessment items, this system aims to facilitate the understanding and application of core cost modeling principles for those new to the field. Initial deployment results will be presented, demonstrating its potential to fill knowledge gaps and improve the ability of engineers to contribute effectively to cost estimation projects.



The chatbot below is built using gradio. It can be used inside this notebook or at the link provided when the interface is generated.

## HuggingFace Account Creation (Free!)

In order to pull an open source model to run this notebook, you will need a HuggingFace account. Once you have an account, you will need to generate an API token.

Steps below:


1.   Create a HuggingFace Account
  * https://huggingface.co/
2.   Follow the steps in the user guide for a user access token (API key)
  * https://huggingface.co/docs/hub/en/security-tokens
3. Paste your user access token / API key into the 2nd cell within the Installation section when prompted



## Installation

In [ ]:
! pip install haystack-ai "transformers>=4.43.1" sentence-transformers accelerate bitsandbytes gradio -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 380.0/380.0 kB 15.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 122.4/122.4 MB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.7/56.7 MB 13.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 319.8/319.8 kB 23.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 94.9/94.9 kB 7.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.0/11.0 MB 76.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.3/73.3 kB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.7/63.7 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.4/89.4 kB 9.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.4/54.4 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 130.2/130.2 kB 11.1 MB/s eta 0:00:00


In [ ]:
import getpass, os
os.environ["HF_API_TOKEN"] = getpass.getpass("Your Hugging Face token")

Your Hugging Face token··········


## Retrieve the pdf files

For pdf files, upload to your google drive, make the link available, and then provide the id found in your link.

Please do not dissiminate the reference PDFs below.

Files other than the ones below can be used. Reference Haystack's Document splitter documentation.

https://docs.haystack.deepset.ai/docs/documentsplitter



In [ ]:
# Required installations
!pip install pdfplumber gdown -q

import os
import pdfplumber
import gdown
from haystack import Document

# Define Google Drive file IDs and local directory
file_info = [
    {"id": "1c7JmBs8ELGQL2R-oSWTnH6bHMz31Zg4b", "name": "nasa_cost_estimating.pdf"},
    {"id": "1sR4sGZzalIvOYkxPnVShxEwc3xLVDSTl", "name": "software_engineering_economics.pdf"},
    {"id": "1vjaaPFs8xKuDyrr_RdYiXFAYbZiRd2At", "name": "cost_and_estimating_guide.pdf"},
    {"id": "12JtQDVk3Q67OBKvPJhEDfYWzZO6_5d3-", "name": "software_engineering.pdf"},
]

local_directory = "./pdf_docs"  # Local directory to store downloaded PDFs

# Ensure the directory exists
os.makedirs(local_directory, exist_ok=True)

# Download each PDF file using gdown
for file in file_info:
    file_id = file["id"]
    file_name = file["name"]
    url = f"https://drive.google.com/uc?id={file_id}"
    output_path = os.path.join(local_directory, file_name)
    gdown.download(url, output=output_path, quiet=False)


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.0/42.0 kB 2.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.5/48.5 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.2/59.2 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.6/5.6 MB 37.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.8/2.8 MB 46.2 MB/s eta 0:00:00


Downloading...
From: https://drive.google.com/uc?id=1c7JmBs8ELGQL2R-oSWTnH6bHMz31Zg4b
To: /content/pdf_docs/nasa_cost_estimating.pdf
100%|██████████| 10.8M/10.8M [00:00<00:00, 63.6MB/s]
Downloading...
From: https://drive.google.com/uc?id=1sR4sGZzalIvOYkxPnVShxEwc3xLVDSTl
To: /content/pdf_docs/software_engineering_economics.pdf
100%|██████████| 38.1M/38.1M [00:00<00:00, 56.6MB/s]
Downloading...
From: https://drive.google.com/uc?id=1vjaaPFs8xKuDyrr_RdYiXFAYbZiRd2At
To: /content/pdf_docs/cost_and_estimating_guide.pdf
100%|██████████| 6.66M/6.66M [00:00<00:00, 44.2MB/s]
Downloading...
From (original): https://drive.google.com/uc?id=12JtQDVk3Q67OBKvPJhEDfYWzZO6_5d3-
From (redirected): https://drive.google.com/uc?id=12JtQDVk3Q67OBKvPJhEDfYWzZO6_5d3-&confirm=t&uuid=9e6ceefc-1692-490d-bc14-bdca7bb64c39
To: /content/pdf_docs/software_engineering.pdf
100%|██████████| 138M/138M [00:01<00:00, 85.8MB/s]


## Create Haystack documents for document store

Parallelization for page processing

In [ ]:
# Function to process a single PDF file and return a Haystack Document
def process_pdf(filename):
    file_path = os.path.join(local_directory, filename)
    full_text = ""

    # Open the PDF and initialize a progress bar for each page
    with pdfplumber.open(file_path) as pdf:
        for page in tqdm(pdf.pages, desc=f"Processing {filename}", unit="page", leave=False):
            text = page.extract_text()
            if text:
                full_text += text + "\n"

    # Return a Haystack Document with extracted content and metadata
    return Document(content=full_text, meta={"title": filename, "source": file_path})


In [ ]:
import os
import pdfplumber
import gdown
from haystack import Document
from tqdm import tqdm
from concurrent.futures import ProcessPoolExecutor

# Step 2: Use ProcessPoolExecutor to process PDFs in parallel
pdf_files = [f for f in os.listdir(local_directory) if f.endswith(".pdf")]

# Initialize the progress bar and ProcessPoolExecutor
raw_docs = []
with ProcessPoolExecutor() as executor:
    # Wrap the executor.map with tqdm for progress visualization
    for doc in tqdm(executor.map(process_pdf, pdf_files), total=len(pdf_files), desc="Processing PDFs", unit="file"):
        raw_docs.append(doc)  # Collect each processed document

# Step 3: Verify the number of documents and a content snippet from the first document
print(f"\nTotal documents loaded: {len(raw_docs)}")  # Display the count of loaded documents
if raw_docs:
    print("\nSample content from first document:")
    print(raw_docs[0].content[:500])  # Print the first 500 characters for verification

Processing PDFs: 100%|██████████| 4/4 [04:42<00:00, 70.59s/file]



Total documents loaded: 4

Sample content from first document:
The image on the cover is the MAVEN spacecraft in orbit around Mars, looking back at Earth. MAVEN launched on
November 18, 2013, and, following a roughly 10-month trip of over 442 million miles, reached Mars on September
21, 2014. The MAVEN project successfully implemented the principles in this handbook to produce their JCL
analysis. MAVEN was launched on schedule and delivered under its budget commitment.
Acknowledgments
Many people contributed to the development of the Version 4.0 edition of 


## Indexing the Pipeline

In [ ]:
from haystack import Pipeline
from haystack.document_stores.in_memory import InMemoryDocumentStore
from haystack import Document
from haystack.components.embedders import SentenceTransformersTextEmbedder, SentenceTransformersDocumentEmbedder
from haystack.components.converters import TextFileToDocument
from haystack.components.writers import DocumentWriter
from haystack.components.preprocessors import DocumentSplitter
from haystack.utils import ComponentDevice

In [ ]:
document_store = InMemoryDocumentStore()

indexing_pipeline = Pipeline()
indexing_pipeline.add_component("splitter", DocumentSplitter(split_by="word", split_length=200))

indexing_pipeline.add_component(
    "embedder",
    SentenceTransformersDocumentEmbedder(
        model="Snowflake/snowflake-arctic-embed-l",  # good embedding model: https://huggingface.co/Snowflake/snowflake-arctic-embed-l
        device=ComponentDevice.from_str("cuda:0"),    # load the model on GPU
    ))
indexing_pipeline.add_component("writer", DocumentWriter(document_store=document_store))

# connect the components
indexing_pipeline.connect("splitter", "embedder")
indexing_pipeline.connect("embedder", "writer")

🚅 Components
  - splitter: DocumentSplitter
  - embedder: SentenceTransformersDocumentEmbedder
  - writer: DocumentWriter
🛤️ Connections
  - splitter.documents -> embedder.documents (List[Document])
  - embedder.documents -> writer.documents (List[Document])

In [ ]:
indexing_pipeline.run({"splitter":{"documents":raw_docs}})

## RAG Pipeline for Standard Convo

In [ ]:
# RAG Prompt Template
from haystack.components.builders import PromptBuilder

prompt_template = """
Using the information contained in the context, provide a comprehensive answer to the question and include your source(s).
If the answer cannot be deduced from the context, inform the user and do not give an answer.

\nContext:
  {% for doc in documents %}
  {{ doc.content }} URL:{{ doc.meta['url'] }}
  {% endfor %};

\nQuestion: {{query}}
\nAnswer:
"""
prompt_builder = PromptBuilder(template=prompt_template)


Here, we use the [`HuggingFaceLocalGenerator`](https://docs.haystack.deepset.ai/docs/huggingfacelocalgenerator), loading the model in Colab with 4-bit quantization.

- meta-llama/Llama-3.2-1B-Instruct
- meta-llama/Meta-Llama-3.1-8B-Instruct

In [ ]:
import torch
from haystack.components.generators import HuggingFaceLocalGenerator

generator = HuggingFaceLocalGenerator(
    model="meta-llama/Meta-Llama-3.1-8B-Instruct",
    huggingface_pipeline_kwargs={"device_map":"auto",
                                  "model_kwargs":{"load_in_4bit":True,
                                                  "bnb_4bit_use_double_quant":True,
                                                  "bnb_4bit_quant_type":"nf4",
                                                  "bnb_4bit_compute_dtype":torch.bfloat16}},
    generation_kwargs={"max_new_tokens": 500})

generator.warm_up()

In [ ]:
from haystack.components.retrievers.in_memory import InMemoryEmbeddingRetriever

query_pipeline = Pipeline()

query_pipeline.add_component(
    "text_embedder",
    SentenceTransformersTextEmbedder(
        model="Snowflake/snowflake-arctic-embed-l",  # good embedding model: https://huggingface.co/Snowflake/snowflake-arctic-embed-l
        device=ComponentDevice.from_str("cuda:0"),  # load the model on GPU
        prefix="Represent this sentence for searching relevant passages: ",  # as explained in the model card (https://huggingface.co/Snowflake/snowflake-arctic-embed-l#using-huggingface-transformers), queries should be prefixed
    ))
query_pipeline.add_component("retriever", InMemoryEmbeddingRetriever(document_store=document_store, top_k=5))
query_pipeline.add_component("prompt_builder", PromptBuilder(template=prompt_template))
query_pipeline.add_component("generator", generator)

# connect the components
query_pipeline.connect("text_embedder.embedding", "retriever.query_embedding")
query_pipeline.connect("retriever.documents", "prompt_builder.documents")
query_pipeline.connect("prompt_builder", "generator")

🚅 Components
  - text_embedder: SentenceTransformersTextEmbedder
  - retriever: InMemoryEmbeddingRetriever
  - prompt_builder: PromptBuilder
  - generator: HuggingFaceLocalGenerator
🛤️ Connections
  - text_embedder.embedding -> retriever.query_embedding (List[float])
  - retriever.documents -> prompt_builder.documents (List[Document])
  - prompt_builder.prompt -> generator.prompt (str)

In [ ]:
def get_generative_answer(query):

  results = query_pipeline.run({
      "text_embedder": {"text": query},
      "prompt_builder": {"query": query}
    }
  )

  answer = results["generator"]["replies"][0]
  rich.print(answer)

In [ ]:
# from IPython.display import Image
# from pprint import pprint
import rich
# import random
# testing the output to make sure it's parseable/following the prompt instructed format
get_generative_answer("tell me about cost modeling with Barry Boehm")

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Setting `pad_token_id` to `eos_token_id`:None for open-end generation.


Barry W. Boehm is a renowned expert in software engineering economics, and his work on cost modeling has had a 
significant impact on the field. His Constructive Cost Model (COCOMO) is a widely used cost estimation model that 
has been applied to over 2000 projects worldwide. COCOMO is a comprehensive model that takes into account various 
factors such as software size, complexity, and development time to estimate costs. Boehm's work on cost modeling 
has been instrumental in helping software developers and managers make informed decisions about software projects.

Boehm's COCOMO model has been extensively used in various industries, including defense, aerospace, and finance. It
has also been used to estimate costs for software projects of varying sizes and complexities. The model has been 
updated over the years, with the latest version being COCOMO II, which was developed in collaboration with other 
researchers.

In addition to COCOMO, Boehm has also developed other cost modeling approaches, such as the PRICE software model 
and the Software Engineering Economics model. These models have been used to estimate costs and make informed 
decisions about software projects.

Boehm's work on cost modeling has been recognized with numerous awards and honors, including the National Medal of 
Technology and Innovation and the IEEE John von Neumann Medal. He has also been elected as a fellow of the National
Academy of Engineering and the American Academy of Arts and Sciences.

In summary, Barry W. Boehm's work on cost modeling has had a significant impact on the field of software 
engineering economics, and his COCOMO model is a widely used and respected cost estimation model.

Sources:

* Boehm, B. W. (1981). Software Engineering Economics. Prentice-Hall.
* Boehm, B. W., & Abts, C. (2000). Software Cost Estimation with COCOMO II. Prentice-Hall.
* Pyster, A. B., & Williams, R. D. (1984). A Software Development Environment for Improving Productivity. IEEE 
Transactions on Software Engineering, 10(4), 429-438.
* Madachy, R., & Selby, R. W. (1995). Cost Models for Future Software Life Cycle Processes: COCOMO 2.0. IEEE 
Transactions on Software Engineering, 21(1), 1-18.

Note: The sources listed above are a selection of Barry Boehm's notable works on cost modeling, and there

## Creating the UI

In [ ]:
import gradio as gr

# the function called when the button is clicked
def get_answer(prompt, history):
    # Call the RAG pipeline to get the response
    results = query_pipeline.run({
        "text_embedder": {"text": prompt},
        "prompt_builder": {"query": prompt}
    })

    # Extract the response and reference section
    response = results["generator"]["replies"][0]
    # reference = "Cost Modeling Document - Section X.Y"  # Example reference format

    # Add the new interaction to the history
    history.append((prompt, response))

    # Format the history for display in reverse order
    history_display = "\n".join([f" **Q:** {q} \n \n \n **A:** {a} \n \n " for q, a in reversed(history)])

    # Wrap the history display with a div for styling
    history_display = f"<div style='border: 2px solid #000; padding: 10px; border-radius: 5px; max-height: 400px; overflow-y: auto;'> {history_display} </div>"

    return response, history_display

# Gradio chatbot interface for Cost Modeling Documents
with gr.Blocks() as costModelingDemo:
    gr.Markdown(
        """
        <img src="https://upload.wikimedia.org/wikipedia/commons/thumb/a/a7/Naval_Postgraduate_School_emblem.svg/250px-Naval_Postgraduate_School_emblem.svg.png" alt="NPS Logo" align="right" width="150" height="103">

        # Cost Modeling Intelligent Chatbot
        ### This chatbot references various cost modeling documents, such as Barry Boehm's works (Software Engineering Economics) and NASA's Cost Estimating Guide. <br><br>
        *NOTE: This chatbot is experimental, and all responses should be verified.*
        """
    )

    # Initialize an empty history
    history = gr.State(value=[])
    with gr.Row():
        with gr.Column(scale=1):
            prompt = gr.Textbox(label="What would you like to know about cost modeling?")
            ask_btn = gr.Button("Ask about Cost Modeling")
            response = gr.Textbox(label="Answer")
            # reference = gr.Textbox(label="Document Section Reference")
        with gr.Column(scale=1):
            # History box
            history_display = gr.Markdown(label="Conversation History", value="")

    ask_btn.click(fn=get_answer, inputs=[prompt, history], outputs=[response, history_display])

costModelingDemo.launch()

# Testable Queries

- Tell me about cost modeling with Barry Boehm.
- Tell me about the COCOMO model.
- What cost factors provide the most leverage to improve productivity?
- What are the tradeoffs between parametric cost models and expert judgment?
- Can you tell me highlights about Barry Boehm's contributions?
- What are the challenges associated with modeling the benefits and value of creating a software product, and how can these be addressed?
- What are the key criteria for evaluating the effectiveness of a cost estimating model (e.g., definition, fidelity, objectivity, etc.)?
- How to determine the optimal time and effort to invest in plans?

[comment]: <> (Off topic:

*   How should one decide on the right amount of testing?
*   How many iterations are recommended for a small web-based prototype?
* How should teams be structured to develop a large system of systems?
* What is IKIWISI and how can it be used?)





<!--
#Barry Boehm Principles

1. Acquisition and development should proceed as risk-driven spiral cycles of increasin

```
# This is formatted as code
```

g elaboration.
1. Process sweet spots can be determined by minimizing risk exposure from counteracting factors (balance anything in terms of risk exposure).
1. For each cycle, identify people's win conditions, project objectives and constraints.
1. Follow the modified golden rule: *Do unto others as you would have others do unto you, if you were like them*.
1. When conflicts arise, prioritize people win conditions.)
-->

# Future Considerations
- Adding memory
  - https://github.com/deepset-ai/haystack-cookbook/blob/main/notebooks/conversational_rag_using_memory.ipynb
- Fine tuning a custom model
  - https://github.com/hiyouga/LLaMA-Factory

<!--
#Barry Boehm Principles

1. Acquisition and development should proceed as risk-driven spiral cycles of increasing elaboration.
1. Process sweet spots can be determined by minimizing risk exposure from counteracting factors (balance anything in terms of risk exposure).
1. For each cycle, identify people's win conditions, project objectives and constraints.
1. Follow the modified golden rule: *Do unto others as you would have others do unto you, if you were like them*.
1. When conflicts arise, prioritize people win conditions.)
-->